###Using Python processing Address-Data
1. first step nee dto import dlt
2. @dlt.table() --> to write data to a streaming table or materialize view based on the way we read data in the function from- if we are using readstream API then the @dlt.table will create a streaming table and if we use only static that is read API then it will create a Materialize view.
3. Last step to create a function that will return a dataframe.

In [0]:
import dlt
from pyspark.sql import functions as F

@dlt.table(
    name = 'bronze_address',
    table_properties = {"quality" : "bronze"},
    comment =' raw layer address'
)
def create_bronze_address ():
  return (
      spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .load('/Volumes/circuitbox/landing/operational_data/addresses/')
    .select(
            "*",
            F.col("_metadata.file_path").alias("input_file_path"),
            F.current_timestamp().alias("ingest_timestamp")))

SIlver layer and Implementing DQ rules

In [0]:
@dlt.table(
    name = 'silver_address_clean',
    table_properties = {"quality" : "silver"},
    comment =' silver layer address'
)

@dlt.expect_or_drop("valid_address", "address_line_1 is not null")
@dlt.expect("valid_postcode", "length(postcode) == 5")
@dlt.expect_or_fail("valid_customer_id", "customer_id is not null")

def create_silver_address_clean ():
  return (
      spark.readStream.table('LIVE.bronze_address')
      .select(
          "customer_id",
          "address_line_1",
          "city",
          "state",
          "postcode",
          F.col("created_date").cast("date")
      )
)

In [0]:
dlt.create_streaming_table(
    name = 'silver_address',
    comment =' silver layer address with SCD type -2',
    table_properties = {"quality" : "silver"}
)

dlt.apply_changes(
    target = 'silver_address',
    source = 'silver_address_clean',
    keys = ['customer_id'],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)